# 1. MILK10k EDA Notebook

Este notebook se va a rehacer poco a poco para centrar el analisis en las variables que interesan para el estudio.

En este primer paso solo cargamos los dos CSV necesarios y construimos un dataframe base para el analisis.

Dataset reference: [ISIC MILK10k Challenge](https://challenge.isic-archive.com/data/#milk10k)


## 2. Construccion del dataframe base

En esta seccion cargamos `MILK10k_Training_Metadata.csv` y `MILK10k_Training_GroundTruth.csv` directamente desde `data/raw`.


In [ ]:
from pathlib import Path

import pandas as pd


In [ ]:
def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / "src" / "milk10k").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root. Launch the notebook from inside the project.")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

train_metadata_path = RAW_DIR / "MILK10k_Training_Metadata.csv"
train_ground_truth_path = RAW_DIR / "MILK10k_Training_GroundTruth.csv"


### 2.1 Carga de tablas

Cargamos solo los dos archivos necesarios para empezar el analisis.


In [ ]:
train_metadata = pd.read_csv(train_metadata_path)
train_ground_truth = pd.read_csv(train_ground_truth_path)

print("train_metadata:", train_metadata.shape)
print("train_ground_truth:", train_ground_truth.shape)


### 2.2 Seleccion de variables y union

Nos quedamos con las columnas de interes del metadata y añadimos todas las clases del ground truth.


In [ ]:
metadata_columns = [
    "lesion_id",
    "image_type",
    "isic_id",
    "age_approx",
    "sex",
    "skin_tone_class",
]

class_columns = [column for column in train_ground_truth.columns if column != "lesion_id"]

df_eda = train_metadata[metadata_columns].merge(
    train_ground_truth[["lesion_id", *class_columns]],
    on="lesion_id",
    how="left",
)

print("df_eda:", df_eda.shape)
df_eda.head()


In [ ]:
import re
import shutil
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit


## 2. Data Setup and Input Files

This section defines the project paths and expected raw files used in the EDA workflow.


In [ ]:
def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / "src" / "milk10k").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root. Launch the notebook from inside the project.")


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
OUTPUT_DIR = DATA_ROOT / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


### 2.1 File Registry

The next cell defines the centralized file map used across the notebook.


In [ ]:
FILE_PATHS = {
    "train_metadata": RAW_DIR / "MILK10k_Training_Metadata.csv",
    "test_metadata": RAW_DIR / "MILK10k_Test_Metadata.csv",
    "train_ground_truth": RAW_DIR / "MILK10k_Training_GroundTruth.csv",
    "train_supplement": RAW_DIR / "MILK10k_Training_Supplement.csv",
    "train_images_zip": RAW_DIR / "MILK10k_Training_Input.zip",
    "test_images_zip": RAW_DIR / "MILK10k_Test_Input.zip",
}


### 2.2 Load Metadata and Annotation Tables

The following cells load metadata and labels, then provide a quick preview of the tables.


In [ ]:
train_metadata = pd.read_csv(FILE_PATHS["train_metadata"])
test_metadata = pd.read_csv(FILE_PATHS["test_metadata"])

print("train_metadata.shape:", train_metadata.shape)
print("test_metadata.shape:", test_metadata.shape)

In [ ]:
display(train_metadata.head(3))

In [ ]:
train_ground_truth = pd.read_csv(FILE_PATHS["train_ground_truth"])
train_supplement = pd.read_csv(FILE_PATHS["train_supplement"])

print("train_ground_truth.shape:", train_ground_truth.shape)
print("train_supplement.shape:", train_supplement.shape)

In [ ]:
display(train_ground_truth.head(3))
display(train_supplement.head(3))

### 2.3 Extract ZIP Archives to `data/processed`

This optional step extracts train/test image ZIP files into `data/processed` with the same folder layout expected by the split pipeline.


In [ ]:
OVERWRITE_PROCESSED = False

zip_extraction_plan = [
    (FILE_PATHS["train_images_zip"], PROCESSED_DIR / "train", "MILK10k_Training_Input"),
    (FILE_PATHS["test_images_zip"], PROCESSED_DIR / "test", "MILK10k_Test_Input"),
]


def extract_dataset_zip(zip_path: Path, destination_parent: Path, expected_root: str, overwrite: bool = False) -> Path:
    # Extract one dataset ZIP while preserving its internal root folder structure.
    destination_parent.mkdir(parents=True, exist_ok=True)
    expected_root_dir = destination_parent / expected_root

    if expected_root_dir.exists() and not overwrite:
        print(f"Skip extraction (already exists): {expected_root_dir}")
        return expected_root_dir

    if expected_root_dir.exists() and overwrite:
        shutil.rmtree(expected_root_dir)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(destination_parent)

    if not expected_root_dir.exists():
        raise FileNotFoundError(f"Expected extracted folder not found: {expected_root_dir}")

    print(f"Extracted: {zip_path.name} -> {expected_root_dir}")
    return expected_root_dir


for zip_path, destination_parent, expected_root in zip_extraction_plan:
    extract_dataset_zip(zip_path, destination_parent, expected_root, overwrite=OVERWRITE_PROCESSED)


### 2.4 Build the Unified Analysis Table

The next cell merges metadata, lesion-level targets, and supplementary image-level annotations into one analysis DataFrame.


In [ ]:
analysis_df = train_metadata.merge(train_ground_truth, on="lesion_id", how="left")
analysis_df = analysis_df.merge(train_supplement, on="isic_id", how="left")

print("analysis_df.shape:", analysis_df.shape)
display(analysis_df.head(3))

# Backward-compatible aliases used in later notebook cells.
milkds = analysis_df
train_gt = train_ground_truth


## 3. Core Metadata EDA

This section explores categorical distributions and key dataset attributes.


### 3.1 Quick Categorical Inspection

This helper prints unique values, counts, and percentages for a selected categorical feature.


In [ ]:
def analyze_categorical(df: pd.DataFrame, col: str) -> None:
    # Summarize one categorical column with counts and relative frequencies.
    print(f"Categorical analysis: {col}")
    print("-" * 50)
    print("Unique values:", df[col].nunique(dropna=False))
    print("Value list:", df[col].drop_duplicates().tolist())
    print("Counts:")
    display(df[col].value_counts(dropna=False).to_frame("count"))
    print("Percentages:")
    display((df[col].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame("percent"))

In [ ]:
analyze_categorical(milkds, "site")

### 3.2 Distribution Plots for Core Attributes

The next cells generate and save bar plots for age, sex, skin tone, and acquisition site.


In [ ]:
bar_output_dir = OUTPUT_DIR / "bar_distributions"
bar_output_dir.mkdir(parents=True, exist_ok=True)

milkds["age_approx"] = milkds["age_approx"].fillna(0).astype(int)
age_order = sorted(milkds["age_approx"].unique())

plt.figure(figsize=(10, 5))
sns.countplot(data=milkds, x="age_approx", order=age_order, palette="pastel")
plt.title("Approximate Age Distribution", fontsize=14)
plt.xlabel("Approximate age (0 = unknown)", fontsize=12)
plt.ylabel("Number of samples", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(bar_output_dir / "bar_age_approx_distribution.png", dpi=150)
plt.show()


In [ ]:
sex_values = milkds["sex"].fillna("unknown")

plt.figure(figsize=(6, 4))
sns.countplot(x=sex_values, order=sex_values.value_counts().index, palette="pastel")
plt.title("Sex Distribution", fontsize=14)
plt.xlabel("Sex")
plt.ylabel("Number of images")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(bar_output_dir / "bar_sex_distribution.png", dpi=150)
plt.show()


In [ ]:
skin_tone_order = [0, 1, 2, 3, 4, 5]

plt.figure(figsize=(8, 5))
sns.countplot(data=milkds, x="skin_tone_class", order=skin_tone_order, palette="copper")
plt.title("Skin Tone Class Distribution", fontsize=14)
plt.xlabel("Skin tone class (0 = unknown)", fontsize=12)
plt.ylabel("Number of images", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(bar_output_dir / "bar_skin_tone_distribution.png", dpi=150)
plt.show()


### 3.2.1 Skin Tone Distribution Across Train/Val/Test Splits

This cell reads the split CSV files (if available) and reports skin tone distribution for `train`, `val`, and `test`.


In [ ]:
split_dir = OUTPUT_DIR / "splits_80_10_10_by_type"
split_files = sorted(split_dir.glob("*_train.csv")) + sorted(split_dir.glob("*_val.csv")) + sorted(split_dir.glob("*_test.csv"))

split_parts = []
for csv_path in split_files:
    df_split = pd.read_csv(csv_path)
    if "skin_tone_class" not in df_split.columns:
        continue

    split_name = csv_path.stem.split("_")[-1].lower()
    image_type = csv_path.stem.rsplit("_", 1)[0]

    temp = df_split[["skin_tone_class"]].copy()
    temp["skin_tone_class"] = pd.to_numeric(temp["skin_tone_class"], errors="coerce")
    temp["split"] = split_name
    temp["image_type"] = image_type
    split_parts.append(temp)

if not split_parts:
    print(
        "No split CSV files found with `skin_tone_class`. "
        "Run section 4.6 first to generate train/val/test split files."
    )
else:
    split_skin_df = pd.concat(split_parts, ignore_index=True)

    split_order = ["train", "val", "test"]
    skin_order = sorted(split_skin_df["skin_tone_class"].dropna().unique().tolist())

    overall_table = (
        split_skin_df
        .groupby(["split", "skin_tone_class"]) 
        .size()
        .reset_index(name="count")
    )
    overall_table["split"] = pd.Categorical(overall_table["split"], categories=split_order, ordered=True)
    overall_table = overall_table.sort_values(["split", "skin_tone_class"]).reset_index(drop=True)

    total_by_split = overall_table.groupby("split")["count"].transform("sum")
    overall_table["percentage"] = (overall_table["count"] / total_by_split * 100).round(2)

    display(overall_table)

    plt.figure(figsize=(10, 4))
    sns.countplot(data=split_skin_df, x="skin_tone_class", hue="split", order=skin_order, hue_order=split_order)
    plt.title("Skin Tone Distribution by Split (Overall)")
    plt.xlabel("Skin tone class")
    plt.ylabel("Number of images")
    plt.tight_layout()
    plt.show()


In [ ]:
site_values = milkds["site"].fillna("unknown")

plt.figure(figsize=(12, 5))
sns.countplot(x=site_values, order=site_values.value_counts().index, palette="pastel")
plt.title("Acquisition Site Distribution", fontsize=14)
plt.xlabel("Acquisition site")
plt.ylabel("Number of images")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(bar_output_dir / "bar_site_distribution.png", dpi=150)
plt.show()


### 3.3 MONET Score Distributions

This step creates boxplots for selected MONET features and saves them to disk.


In [ ]:
def boxplot_monet(df: pd.DataFrame, col: str, out_path: Path) -> None:
    # Create a single-feature boxplot to inspect MONET score spread and outliers.
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[col], palette="pastel")
    plt.title(f"Boxplot of {col}", fontsize=14)
    plt.xlabel(col, fontsize=12)
    plt.grid(axis="x", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()


In [ ]:
monet_output_dir = OUTPUT_DIR / "monet_boxplot"
monet_output_dir.mkdir(parents=True, exist_ok=True)

monet_columns = [
    "MONET_ulceration_crust",
    "MONET_hair",
    "MONET_vasculature_vessels",
    "MONET_erythema",
    "MONET_pigmented",
    "MONET_gel_water_drop_fluid_dermoscopy_liquid",
    "MONET_skin_markings_pen_ink_purple_pen",
]

for monet_col in monet_columns:
    boxplot_monet(milkds, monet_col, monet_output_dir / f"{monet_col}.png")


### 3.4 Ground Truth Class Distribution

Finally, this cell visualizes lesion-level class counts from the one-hot ground truth table.


In [ ]:
class_cols = [c for c in train_gt.columns if c != "lesion_id"]
class_counts = train_gt[class_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="crest")
plt.title("MILK10k Training Ground Truth Class Distribution", fontsize=14)
plt.xlabel("Diagnostic class")
plt.ylabel("Number of lesions")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.6)

out_path = OUTPUT_DIR / "bar_groundtruth_classes.png"
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(out_path, dpi=150)
plt.show()


## 4. Dataset Split Preparation

This section prepares lesion-level labels, maps image paths, creates stratified splits, and exports split files.


### 4.1 Define Inputs for Split Generation

The next cell defines training image and CSV sources using the paths configured at the top of the notebook.


In [ ]:
TRAIN_IMG_ROOT = PROCESSED_DIR / "train" / "MILK10k_Training_Input"
META_CSV = RAW_DIR / "MILK10k_Training_Metadata.csv"
GT_CSV = RAW_DIR / "MILK10k_Training_GroundTruth.csv"


### 4.2 Standardize Metadata and Ground Truth Schemas

This step makes column names consistent (`isic_id`, `lesion_id`, `image_type`) and detects one-hot target columns.


In [ ]:
def load_and_standardize_metadata(meta_csv: Path) -> pd.DataFrame:
    # Normalize common metadata column variants to a shared schema.
    df = pd.read_csv(meta_csv)
    cols = {c.lower(): c for c in df.columns}

    isic_col = cols.get("isic_id") or cols.get("image_id") or cols.get("isic")
    lesion_col = cols.get("lesion_id") or cols.get("lesion") or cols.get("il_id") or cols.get("il")
    image_type_col = cols.get("image_type") or cols.get("type") or cols.get("img_type")

    if not isic_col or not lesion_col:
        raise ValueError(f"Missing isic_id/lesion_id columns in {meta_csv.name}. Found: {df.columns.tolist()}")

    if image_type_col is None:
        image_type_col = "image_type"
        df[image_type_col] = "unknown"

    df = df.rename(columns={isic_col: "isic_id", lesion_col: "lesion_id", image_type_col: "image_type"})
    df["isic_id"] = df["isic_id"].astype(str)
    df["lesion_id"] = df["lesion_id"].astype(str)
    df["image_type"] = df["image_type"].astype(str).str.strip().str.lower()
    return df


def load_and_standardize_groundtruth(gt_csv: Path):
    # Detect the key column and return label columns for one-hot targets.
    df = pd.read_csv(gt_csv)
    cols = {c.lower(): c for c in df.columns}

    key_col = None
    for candidate in ["lesion_id", "isic_id", "image_id", "lesion", "isic"]:
        if candidate in cols:
            key_col = cols[candidate]
            break

    if key_col is None:
        raise ValueError(f"No join key found in {gt_csv.name} (expected lesion_id or isic_id variants)")

    df = df.rename(columns={key_col: "key"})
    non_label_cols = ["key", "isic_id", "lesion_id", "image_id", "split"]
    label_cols = [c for c in df.columns if c not in non_label_cols]

    for col in label_cols:
        values = set(pd.Series(df[col]).dropna().unique().tolist())
        if values.issubset({0, 1, 0.0, 1.0}):
            df[col] = df[col].astype(int)

    return df, label_cols


meta_df = load_and_standardize_metadata(META_CSV)
gt_df, label_cols = load_and_standardize_groundtruth(GT_CSV)

print("Metadata shape:", meta_df.shape)
print("Ground truth shape:", gt_df.shape)
print("First labels:", label_cols[:8])
display(meta_df.head(3))
display(gt_df.head(3))


### 4.3 Build Lesion-Level Labels

The following cell converts labels to lesion level and derives `class_idx`/`class_name` for valid one-hot rows.


In [ ]:
gt_is_lesion_level = gt_df["key"].isin(meta_df["lesion_id"]).mean() > 0.5

if gt_is_lesion_level:
    gt_lesion = gt_df.rename(columns={"key": "lesion_id"}).copy()
else:
    # If labels are image-level, map image IDs to lesion IDs and aggregate by lesion.
    id_to_lesion = meta_df.set_index("isic_id")["lesion_id"].to_dict()
    gt_image_level = gt_df.copy()
    gt_image_level["lesion_id"] = gt_image_level["key"].map(id_to_lesion)
    gt_lesion = (
        gt_image_level.drop(columns=["key"])
        .groupby("lesion_id", as_index=False)[label_cols]
        .max()
    )

label_matrix = gt_lesion[label_cols].to_numpy()
ones_per_row = label_matrix.sum(axis=1)
valid_single_label = ones_per_row == 1

class_idx = np.where(valid_single_label, label_matrix.argmax(axis=1), -1)
idx_to_name = {i: class_name for i, class_name in enumerate(label_cols)}

gt_lesion["class_idx"] = class_idx
gt_lesion["class_name"] = gt_lesion["class_idx"].map(idx_to_name)

display(gt_lesion[["lesion_id", "class_idx", "class_name"]].head(5))


### 4.4 Attach Image Paths and Filter Missing Files

This step maps each row to an image path and keeps only rows that exist on disk.


In [ ]:
def find_image_path_for_isic(isic_id: str, lesion_id: str, root: Path):
    # Resolve image file path from the expected folder pattern root/lesion_id/isic_id.*
    lesion_dir = root / lesion_id
    if not lesion_dir.exists():
        return None

    candidates = list(lesion_dir.glob(f"{isic_id}.*"))
    if not candidates:
        candidates = list(lesion_dir.glob(f"*{isic_id}*"))

    return candidates[0] if candidates else None


master = meta_df.merge(gt_lesion[["lesion_id", "class_idx", "class_name"]], on="lesion_id", how="left")

image_paths = []
exists_flags = []
for row in master.itertuples(index=False):
    path = find_image_path_for_isic(row.isic_id, row.lesion_id, TRAIN_IMG_ROOT)
    image_paths.append(path)
    exists_flags.append(path is not None and Path(path).exists())

master["img_path"] = image_paths
master["exists"] = exists_flags
master = master[master["exists"]].drop(columns=["exists"]).copy()

print("Master shape after path filtering:", master.shape)
display(master[["isic_id", "lesion_id", "image_type", "img_path"]].head(5))


### 4.5 Stratified Lesion-Level Split (80/10/10)

The next function avoids lesion leakage and stratifies by `(class_idx, skin_tone_class)` when possible.


In [ ]:
def stratified_split_by_lesion_label_and_skin(
    master_df: pd.DataFrame,
    random_state: int = 42,
    min_stratum_count: int = 5,
    train_size: float = 0.8,
    val_size: float = 0.1,
    test_size: float = 0.1,
):
    # Create lesion-level splits, then project split membership back to image rows.
    if not np.isclose(train_size + val_size + test_size, 1.0):
        raise ValueError("Split sizes must sum to 1.0")

    lesion_tbl = (
        master_df.groupby("lesion_id", as_index=False)
        .agg({"class_idx": "first", "skin_tone_class": "first"})
    )
    lesion_tbl["skin_tone_class"] = lesion_tbl["skin_tone_class"].fillna(0).astype(int)
    lesion_tbl["stratum"] = lesion_tbl["class_idx"].astype(str) + "_" + lesion_tbl["skin_tone_class"].astype(str)

    stratum_counts = lesion_tbl["stratum"].value_counts()
    rare_strata = stratum_counts[stratum_counts < min_stratum_count].index
    if len(rare_strata) > 0:
        rare_mask = lesion_tbl["stratum"].isin(rare_strata)
        lesion_tbl.loc[rare_mask, "stratum"] = lesion_tbl.loc[rare_mask, "class_idx"].astype(str) + "_other"

    if (lesion_tbl["stratum"].value_counts() < 2).any():
        print("Warning: fallback to class-only stratification due to tiny strata.")
        lesion_tbl["stratum"] = lesion_tbl["class_idx"].astype(str)

    lesion_ids = lesion_tbl["lesion_id"].values
    strata = lesion_tbl["stratum"].values

    temp_size = 1.0 - train_size
    sss_train = StratifiedShuffleSplit(n_splits=1, test_size=temp_size, random_state=random_state)
    train_idx, temp_idx = next(sss_train.split(lesion_ids, strata))

    lesion_train = set(lesion_tbl.iloc[train_idx]["lesion_id"])
    lesion_temp = lesion_tbl.iloc[temp_idx].reset_index(drop=True)

    val_ratio_in_temp = val_size / (val_size + test_size)
    sss_val_test = StratifiedShuffleSplit(
        n_splits=1,
        test_size=(1.0 - val_ratio_in_temp),
        random_state=random_state,
    )
    val_idx, test_idx = next(sss_val_test.split(lesion_temp["lesion_id"].values, lesion_temp["stratum"].values))

    lesion_val = set(lesion_temp.iloc[val_idx]["lesion_id"])
    lesion_test = set(lesion_temp.iloc[test_idx]["lesion_id"])

    splits = {}
    for image_type, subset in master_df.groupby("image_type"):
        splits[image_type] = {
            "train": subset[subset["lesion_id"].isin(lesion_train)].reset_index(drop=True),
            "val": subset[subset["lesion_id"].isin(lesion_val)].reset_index(drop=True),
            "test": subset[subset["lesion_id"].isin(lesion_test)].reset_index(drop=True),
        }
        print(
            f"[{image_type}] train={len(splits[image_type]['train'])}, "
            f"val={len(splits[image_type]['val'])}, test={len(splits[image_type]['test'])}"
        )

    return splits, (lesion_train, lesion_val, lesion_test)


splits, lesion_sets = stratified_split_by_lesion_label_and_skin(
    master,
    random_state=123,
    min_stratum_count=5,
    train_size=0.8,
    val_size=0.1,
    test_size=0.1,
)

print("Image types:", list(splits.keys()))


### 4.6 Export Split CSV Files

This cell writes one CSV per `image_type` and split (`train`, `val`, `test`).


In [ ]:
OUT_SPLIT_DIR = OUTPUT_DIR / "splits_80_10_10_by_type"
OUT_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

required_cols = [
    "img_path",
    "class_idx",
    "class_name",
    "lesion_id",
    "isic_id",
    "image_type",
    "skin_tone_class",
]

for image_type, split_dict in splits.items():
    for split_name, split_df in split_dict.items():
        available_cols = [c for c in required_cols if c in split_df.columns]
        split_df[available_cols].to_csv(OUT_SPLIT_DIR / f"{image_type}_{split_name}.csv", index=False)

print("Saved split CSV files to:", OUT_SPLIT_DIR)


### 4.7 Optional Export: Folder Layout by Class

The next cell copies images into a class-based folder structure for each split and image type.


In [ ]:
FINAL_DATASETS_DIR = DATA_ROOT / "final_datasets_by_class"
FINAL_DATASETS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF",
    "MAL_OTH", "MEL", "NV", "SCCKA", "VASC",
]


def reorganize_by_class(image_type_raw: str) -> None:
    # Copy split images into train/val/test/<class_name>/ for downstream training pipelines.
    image_type_clean = image_type_raw.strip().lower()
    if "clinical" in image_type_clean:
        image_type_clean = "clinical"
    elif "dermoscopic" in image_type_clean:
        image_type_clean = "dermoscopic"
    else:
        image_type_clean = re.sub(r"[^a-z0-9]+", "_", image_type_clean)

    dataset_dir = FINAL_DATASETS_DIR / f"DATASET_{image_type_clean.upper()}"
    dataset_dir.mkdir(parents=True, exist_ok=True)

    split_files = {
        "train": OUT_SPLIT_DIR / f"{image_type_raw}_train.csv",
        "val": OUT_SPLIT_DIR / f"{image_type_raw}_val.csv",
        "test": OUT_SPLIT_DIR / f"{image_type_raw}_test.csv",
    }

    for split_name, csv_path in split_files.items():
        if not csv_path.exists():
            print(f"Missing split file: {csv_path.name}")
            continue

        split_df = pd.read_csv(csv_path)
        split_dir = dataset_dir / split_name
        split_dir.mkdir(parents=True, exist_ok=True)

        for class_name in CLASS_NAMES:
            (split_dir / class_name).mkdir(parents=True, exist_ok=True)

        for row in split_df.itertuples(index=False):
            src = Path(row.img_path)
            dst = split_dir / row.class_name / src.name
            shutil.copy2(src, dst)

    print(f"Exported class-based folder structure for: {image_type_raw}")


for image_type in splits.keys():
    reorganize_by_class(image_type)


### 4.8 Split Quality Checks

This section visualizes class and skin tone distributions across splits (overall and per image type).


In [ ]:
all_parts = []
for image_type, split_dict in splits.items():
    for split_name, split_df in split_dict.items():
        temp_df = split_df.copy()
        temp_df["split"] = split_name
        temp_df["image_type"] = image_type
        all_parts.append(temp_df)

splits_df = pd.concat(all_parts, ignore_index=True)

plt.figure(figsize=(14, 5))
sns.countplot(
    data=splits_df,
    x="class_name",
    hue="split",
    order=sorted(splits_df["class_name"].dropna().unique()),
)
plt.title("Class distribution by split (overall)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

g = sns.catplot(
    data=splits_df,
    x="class_name",
    hue="split",
    col="image_type",
    kind="count",
    col_wrap=2,
    height=4,
    aspect=1.6,
    order=sorted(splits_df["class_name"].dropna().unique()),
)
g.set_xticklabels(rotation=45, ha="right")
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle("Class distribution by split (per image type)")
plt.show()

skin_order = sorted(splits_df["skin_tone_class"].dropna().unique().tolist())

plt.figure(figsize=(10, 4))
sns.countplot(data=splits_df, x="skin_tone_class", hue="split", order=skin_order)
plt.title("Skin tone distribution by split (overall)")
plt.tight_layout()
plt.show()

g = sns.catplot(
    data=splits_df,
    x="skin_tone_class",
    hue="split",
    col="image_type",
    kind="count",
    col_wrap=2,
    height=4,
    aspect=1.6,
    order=skin_order,
)
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle("Skin tone distribution by split (per image type)")
plt.show()


### 4.9 Paired Image Grid by Skin Tone

The final check displays paired clinical/dermoscopic images per skin tone to inspect visual balance.


In [ ]:
def plot_all_pairs_single_grid(
    master_df: pd.DataFrame,
    skin_tones=(0, 1, 2, 3, 4, 5),
    n_pairs: int = 5,
    seed: int = 42,
):
    import numpy as np
    import matplotlib.pyplot as plt
    from PIL import Image

    # ---------- Build paired table ----------
    pivot = (
        master_df.pivot_table(
            index=["lesion_id", "skin_tone_class"],
            columns="image_type",
            values="img_path",
            aggfunc="first",
        )
        .reset_index()
    )

    image_type_cols = [c for c in pivot.columns if isinstance(c, str)]
    derm_col = next((c for c in image_type_cols if "derm" in c.lower()), None)
    clin_col = next((c for c in image_type_cols if "clin" in c.lower()), None)

    if derm_col is None or clin_col is None:
        if len(image_type_cols) < 2:
            raise ValueError("Not enough image_type columns to form pairs")
        derm_col, clin_col = image_type_cols[:2]

    pivot = pivot.dropna(subset=[derm_col, clin_col]).copy()

    # ---------- Paper-like style ----------
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 20,
        "axes.titlesize": 32,
        "figure.titlesize": 36,
    })

    n_rows = len(skin_tones)
    n_cols = 2 * n_pairs

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(2.6 * n_cols, 2.8 * n_rows),
        dpi=180,
        constrained_layout=True,
    )

    if n_rows == 1:
        axes = np.array([axes])
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    fig.suptitle(
        f"Paired images ({derm_col} vs {clin_col})",
        y=1.08,
        fontweight="semibold",
    )

    for row_idx, skin_tone in enumerate(skin_tones):
        subset = pivot[pivot["skin_tone_class"] == skin_tone]

        if len(subset) == 0:
            for col_idx in range(n_cols):
                axes[row_idx, col_idx].axis("off")

            axes[row_idx, 0].text(
                -0.1,
                0.5,
                f"Skin tone {skin_tone}",
                transform=axes[row_idx, 0].transAxes,
                fontsize=36,
                fontweight="semibold",
                rotation=90,
                va="center",
                ha="center",
            )

            axes[row_idx, 0].text(
                0.5,
                0.5,
                "No pairs",
                transform=axes[row_idx, 0].transAxes,
                fontsize=20,
                ha="center",
                va="center",
            )
            continue

        sample_size = min(n_pairs, len(subset))
        sampled = subset.sample(sample_size, random_state=seed)

        for pair_idx in range(n_pairs):
            left_col = 2 * pair_idx
            right_col = left_col + 1

            left_ax = axes[row_idx, left_col]
            right_ax = axes[row_idx, right_col]

            if pair_idx >= sample_size:
                left_ax.axis("off")
                right_ax.axis("off")
                continue

            row = sampled.iloc[pair_idx]
            derm_img = Image.open(row[derm_col]).convert("RGB")
            clin_img = Image.open(row[clin_col]).convert("RGB")

            left_ax.imshow(derm_img)
            right_ax.imshow(clin_img)

            left_ax.set_xticks([])
            left_ax.set_yticks([])
            right_ax.set_xticks([])
            right_ax.set_yticks([])

            for spine in left_ax.spines.values():
                spine.set_visible(False)
            for spine in right_ax.spines.values():
                spine.set_visible(False)

            if row_idx == 0:
                left_ax.set_title("Dermoscopic", fontsize=26, fontweight="semibold", pad=15)
                right_ax.set_title("Clinical", fontsize=26, fontweight="semibold", pad=15)

        # Larger, cleaner row label
        axes[row_idx, 0].text(
            -0.15,
            0.5,
            f"Skin tone {skin_tone}",
            transform=axes[row_idx, 0].transAxes,
            fontsize=26,
            fontweight="semibold",
            rotation=90,
            va="center",
            ha="center",
        )

    plt.show()

plot_all_pairs_single_grid(master, skin_tones=(0, 1, 2, 3, 4, 5), n_pairs=5, seed=42)

## 5. Paper-Ready EDA Figures

This final section generates high-resolution figures suitable for the thesis document. All figures are saved to `figures/eda` using a consistent visual style.


### 5.1 Figure Style and Output Directory

The following cell defines a publication-style theme and the output directory used to export the figures.


In [ ]:
FIGURES_EDA_DIR = PROJECT_ROOT / "figures" / "eda"
FIGURES_EDA_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 320,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Times"],
})

PAPER_COLORS = {
    "clinical": "#4E5D6C",
    "dermoscopic": "#7A8471",
    "benign": "#B8C4B1",
    "malignant": "#7A4A44",
    "skin": "#8E6C4C",
    "sex": "#6F86A6",
    "age": "#5E6D7A",
}


def save_paper_figure(fig, name: str):
    out_path = FIGURES_EDA_DIR / name
    fig.savefig(out_path, facecolor="white")
    print(f"Saved: {out_path}")
    return out_path


print(f"Figure output directory: {FIGURES_EDA_DIR}")


### 5.2 Paired Clinical and Dermoscopic Examples

This figure shows representative paired samples across skin-tone groups to highlight modality differences and visual variability.


In [ ]:
paired_df = (
    master.pivot_table(
        index=["lesion_id", "skin_tone_class", "class_name"],
        columns="image_type",
        values="img_path",
        aggfunc="first",
    )
    .reset_index()
)

pair_cols = [c for c in paired_df.columns if isinstance(c, str)]
derm_col = next((c for c in pair_cols if "derm" in c.lower()), None)
clin_col = next((c for c in pair_cols if "clin" in c.lower()), None)
if derm_col is None or clin_col is None:
    raise ValueError("Could not identify clinical and dermoscopic columns in paired dataframe.")

# Prefer cleaner-looking lesion categories for a thesis figure, then rank candidates
# within each skin tone using simple visual heuristics to avoid particularly harsh images.
preferred_classes = ["NV", "BKL", "DF", "MEL", "VASC", "AKIEC", "BCC", "SCCKA", "BEN_OTH", "INF", "MAL_OTH"]
class_rank = {name: idx for idx, name in enumerate(preferred_classes)}

# Manual final selection for the paper figure. These lesion IDs were chosen to keep
# the paired examples visually clearer while preserving all six skin-tone rows.
selected_lesions_by_skin = {
    0: ["IL_8962270", "IL_4618514", "IL_7167860"],
    1: ["IL_4933639", "IL_4456737", "IL_8331769"],
    2: ["IL_1664257", "IL_0127859", "IL_4732392"],
    3: ["IL_1355090", "IL_3162928", "IL_1024518"],
    4: ["IL_5961668", "IL_2371687", "IL_0682550"],
    5: ["IL_6393751", "IL_4962584", "IL_9481592"],
}

def pair_visual_score(clin_path, derm_path):
    def image_score(img_path):
        arr = np.asarray(Image.open(img_path).convert("RGB").resize((160, 160)), dtype=np.float32)
        brightness = arr.mean()
        redness = np.clip(arr[..., 0] - 0.5 * (arr[..., 1] + arr[..., 2]), 0, None).mean()
        saturation = arr.std(axis=2).mean()
        darkness_penalty = abs(brightness - 168.0)
        return 0.55 * redness + 0.30 * saturation + 0.15 * darkness_penalty

    return 0.45 * image_score(clin_path) + 0.55 * image_score(derm_path)

def select_presentable_pairs(subset, n_pairs):
    ranked = subset.copy()
    ranked["class_rank"] = ranked["class_name"].map(lambda x: class_rank.get(x, len(class_rank)))
    ranked["visual_score"] = ranked.apply(lambda r: pair_visual_score(r[clin_col], r[derm_col]), axis=1)
    ranked = ranked.sort_values(["class_rank", "visual_score", "lesion_id"]).reset_index(drop=True)
    return ranked.head(n_pairs)

def choose_pairs_for_skin(subset, skin_tone, n_pairs):
    selected_ids = selected_lesions_by_skin.get(skin_tone, [])
    manual = subset[subset["lesion_id"].isin(selected_ids)].copy()
    if len(manual) > 0:
        manual["pair_order"] = manual["lesion_id"].map({lid: idx for idx, lid in enumerate(selected_ids)})
        manual = manual.sort_values("pair_order").drop(columns="pair_order")
    if len(manual) >= n_pairs:
        return manual.head(n_pairs)
    fallback = select_presentable_pairs(subset[~subset["lesion_id"].isin(selected_ids)], n_pairs - len(manual))
    return pd.concat([manual, fallback], ignore_index=True)

skin_levels = [0, 1, 2, 3, 4, 5]
pairs_per_row = 3
ncols = pairs_per_row * 3 - 1
width_ratios = []
for pair_idx in range(pairs_per_row):
    width_ratios.extend([1, 1])
    if pair_idx < pairs_per_row - 1:
        width_ratios.append(0.36)

fig, axes = plt.subplots(
    len(skin_levels),
    ncols,
    figsize=(18.2, 15.8),
    facecolor="white",
    gridspec_kw={"width_ratios": width_ratios},
)

for row_idx, skin_tone in enumerate(skin_levels):
    subset = paired_df[(paired_df["skin_tone_class"] == skin_tone)].dropna(subset=[derm_col, clin_col])
    sampled = choose_pairs_for_skin(subset, skin_tone, pairs_per_row) if len(subset) > 0 else pd.DataFrame(columns=subset.columns)
    sample_size = len(sampled)

    col_ptr = 0
    for pair_idx in range(pairs_per_row):
        ax_clin = axes[row_idx, col_ptr]
        ax_derm = axes[row_idx, col_ptr + 1]

        for ax in [ax_clin, ax_derm]:
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_facecolor("#F7F5F2")
            for spine in ax.spines.values():
                spine.set_visible(False)

        if pair_idx < sample_size:
            row = sampled.iloc[pair_idx]
            ax_clin.imshow(Image.open(row[clin_col]).convert("RGB"))
            ax_derm.imshow(Image.open(row[derm_col]).convert("RGB"))
            if row_idx == 0:
                ax_clin.set_title("Clinical", fontsize=26, pad=10, fontweight="bold")
                ax_derm.set_title("Dermoscopic", fontsize=26, pad=10, fontweight="bold")
        else:
            ax_clin.axis("off")
            ax_derm.axis("off")

        col_ptr += 2
        if pair_idx < pairs_per_row - 1:
            axes[row_idx, col_ptr].axis("off")
            col_ptr += 1

    axes[row_idx, 0].set_ylabel(
        f"Skin tone \n{skin_tone}",
        fontsize=30,
        fontweight="bold",
        rotation=90,
        labelpad=34,
    )
    axes[row_idx, 0].yaxis.set_label_coords(-0.12, 0.5)

plt.subplots_adjust(left=0.11, right=0.98, top=0.97, bottom=0.03, wspace=0.06, hspace=0.24)
save_paper_figure(fig, "paired_examples_clinical_dermoscopic.png")
plt.show()


### 5.3 Overall Label Distributions

The following figure summarizes the 11-class and binary label imbalance using all available images together, without separating by sensor or split.


In [ ]:
counts_11f = (
    splits_df.groupby("class_name")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

malignant_classes = {"AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"}
counts_2f = splits_df.copy()
counts_2f["binary_label"] = np.where(
    counts_2f["class_name"].isin(malignant_classes),
    "malignant",
    "benign",
)
counts_2f = (
    counts_2f.groupby("binary_label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

def annotate_bar_counts(ax, fontsize=18, offset_ratio=0.012):
    visible_heights = [patch.get_height() for patch in ax.patches if patch.get_height() > 0]
    if not visible_heights:
        return
    y_max = max(visible_heights)
    for patch in ax.patches:
        height = patch.get_height()
        if height <= 0:
            continue
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            height + y_max * offset_ratio,
            f"{int(height):,}",
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="#3B3B3B",
        )
    ax.set_ylim(0, y_max * 1.12)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(18.0, 7.2),
    facecolor="white",
    gridspec_kw={"width_ratios": [2.4, 1.1]},
    constrained_layout=True,
)

sns.barplot(
    data=counts_11f,
    x="class_name",
    y="count",
    color=PAPER_COLORS["dermoscopic"],
    edgecolor="#3F4A44",
    linewidth=0.6,
    ax=axes[0],
)
axes[0].set_xlabel("Class", fontsize=30, labelpad=16)
axes[0].set_ylabel("Number of images", fontsize=30, labelpad=16)
axes[0].tick_params(axis="x", rotation=38, labelsize=25)
axes[0].tick_params(axis="y", labelsize=25)
axes[0].ticklabel_format(style="plain", axis="y")
axes[0].locator_params(axis="y", nbins=4)
annotate_bar_counts(axes[0], fontsize=24, offset_ratio=0.01)

sns.barplot(
    data=counts_2f,
    x="binary_label",
    y="count",
    order=["malignant", "benign"],
    palette=[PAPER_COLORS["malignant"], PAPER_COLORS["benign"]],
    edgecolor="#4E4E4E",
    linewidth=0.6,
    ax=axes[1],
)
for patch in axes[1].patches:
    original_width = patch.get_width()
    new_width = 0.30
    patch.set_width(new_width)
    patch.set_x(patch.get_x() + (original_width - new_width) / 2)

axes[1].set_xlabel("Class", fontsize=30, labelpad=16)
axes[1].set_ylabel("Number of images", fontsize=30, labelpad=16)
axes[1].tick_params(axis="x", labelsize=25)
axes[1].tick_params(axis="y", labelsize=25)
axes[1].ticklabel_format(style="plain", axis="y")
axes[1].locator_params(axis="y", nbins=4)
annotate_bar_counts(axes[1], fontsize=24, offset_ratio=0.02)

for ax in axes:
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.8, alpha=1)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#C8C8C8")
    ax.spines["bottom"].set_color("#C8C8C8")

save_paper_figure(fig, "class_distribution_11f_2f_overall.png")
plt.show()


### 5.4 Metadata Distributions

This figure summarizes the distributions of skin tone, sex, and age in a single horizontal layout.


In [ ]:
meta_plot_df = train_metadata.copy()
meta_plot_df["skin_tone_class"] = meta_plot_df["skin_tone_class"].fillna(0).astype(int)
meta_plot_df["sex"] = meta_plot_df["sex"].fillna("Unknown").astype(str).str.title()

def annotate_bar_counts(ax, fontsize=16, offset_ratio=0.012):
    visible_heights = [patch.get_height() for patch in ax.patches if patch.get_height() > 0]
    if not visible_heights:
        return
    y_max = max(visible_heights)
    for patch in ax.patches:
        height = patch.get_height()
        if height <= 0:
            continue
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            height + y_max * offset_ratio,
            f"{int(height):,}",
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="#3B3B3B",
        )
    ax.set_ylim(0, y_max * 1.14)

def annotate_hist_counts(ax, fontsize=14, offset_ratio=0.012):
    visible_heights = [patch.get_height() for patch in ax.patches if patch.get_height() > 0]
    if not visible_heights:
        return
    y_max = max(visible_heights)
    for patch in ax.patches:
        height = patch.get_height()
        if height <= 0:
            continue
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            height + y_max * offset_ratio,
            f"{int(height):,}",
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="#3B3B3B",
        )
    ax.set_ylim(0, y_max * 1.16)

fig, axes = plt.subplots(1, 3, figsize=(18.5, 5.8), facecolor="white", constrained_layout=True)

# -----------------------------
# Skin tone distribution
# -----------------------------
skin_counts = (
    meta_plot_df["skin_tone_class"]
    .value_counts()
    .sort_index()
    .rename_axis("skin_tone_class")
    .reset_index(name="count")
)

skin_palette = sns.color_palette("copper", n_colors=len(skin_counts))

sns.barplot(
    data=skin_counts,
    x="skin_tone_class",
    y="count",
    palette=skin_palette,
    edgecolor="#4F3B2C",
    linewidth=1.0,
    ax=axes[0],
)

axes[0].set_xlabel("Skin tone class", fontsize=26, labelpad=14)
axes[0].set_ylabel("Number of cases", fontsize=26, labelpad=14)
axes[0].tick_params(axis="x", labelsize=25)
axes[0].tick_params(axis="y", labelsize=25)
axes[0].ticklabel_format(style="plain", axis="y")
axes[0].locator_params(axis="y", nbins=4)
annotate_bar_counts(axes[0], fontsize=22, offset_ratio=0.014)

# -----------------------------
# Sex distribution
# -----------------------------
sex_order = meta_plot_df["sex"].value_counts().index.tolist()
sex_palette = sns.color_palette("pastel", n_colors=len(sex_order))
sex_palette = [sns.desaturate(c, 0.9) for c in sex_palette]

sns.countplot(
    data=meta_plot_df,
    x="sex",
    order=sex_order,
    palette=sex_palette,
    edgecolor="#5A5A5A",
    linewidth=1.0,
    ax=axes[1],
)

for patch in axes[1].patches:
    original_width = patch.get_width()
    new_width = 0.38
    patch.set_width(new_width)
    patch.set_x(patch.get_x() + (original_width - new_width) / 2)

axes[1].set_xlabel("Sex", fontsize=26, labelpad=14)
axes[1].set_ylabel("Number of cases", fontsize=26, labelpad=14)
axes[1].tick_params(axis="x", labelsize=25)
axes[1].tick_params(axis="y", labelsize=25)
axes[1].ticklabel_format(style="plain", axis="y")
axes[1].locator_params(axis="y", nbins=4)
annotate_bar_counts(axes[1], fontsize=22, offset_ratio=0.014)

# -----------------------------
# Age distribution
# -----------------------------
age_palette = sns.color_palette("copper", n_colors=6)
age_color = sns.color_palette("muted")[0]

age_bins = np.arange(0, 91, 10)

sns.histplot(
    data=meta_plot_df,
    x="age_approx",
    bins=age_bins,
    color=age_color,
    edgecolor="#000000",
    linewidth=0.9,
    alpha=0.9,
    shrink=0.8,
    ax=axes[2],
)

axes[2].set_xticks(age_bins)

age_min = int(np.floor(meta_plot_df["age_approx"].min() / 10.0) * 10)
age_max = int(np.ceil(meta_plot_df["age_approx"].max() / 10.0) * 10)
axes[2].set_xticks(np.arange(age_min, age_max + 1, 10))

axes[2].set_xlabel("Age", fontsize=26, labelpad=14)
axes[2].set_ylabel("Number of cases", fontsize=26, labelpad=14)
axes[2].tick_params(axis="x", labelsize=25)
axes[2].tick_params(axis="y", labelsize=25)
axes[2].ticklabel_format(style="plain", axis="y")
axes[2].locator_params(axis="y", nbins=4)
annotate_hist_counts(axes[2], fontsize=20, offset_ratio=0.012)

# -----------------------------
# Shared styling
# -----------------------------
for ax in axes:
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.8, alpha=0.8)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#C8C8C8")
    ax.spines["bottom"].set_color("#C8C8C8")

save_paper_figure(fig, "metadata_distribution_skin_sex_age.png")
plt.show()